# Python Code

In [45]:
from numpy.random import rand
import sys
import numpy as np
from tqdm import tqdm
from memory_profiler import memory_usage as mmu
import time

import random, sys
from numba import jit, njit, prange

import xarray as xr
import plotly.express as py
import pandas as pd

import plotly.graph_objects as go


In [46]:
# Python code - loop

def calc_pi_loop(n):
    h = 0 # Number of hits inside the circle
    
    for _ in range(n):
        x, y = rand(), rand() # Random points in [0, 1)
    
        if x*x + y*y < 1.:
            h += 1 # Successful hit
        
    return 4. * float(h) / float(n) # Estimate pi

In [47]:
# Numpy code

def calc_pi_numpy(n):
    
    h = sum(rand(n)**2 + rand(n)**2 < 1.)
        
    return 4. * float(h) / float(n) # Estimate pi

In [48]:
# Numba codes - normal and parallel one

@njit
def calc_pi_numba(n):
    h = 0
    for _ in range(n):
        x = random.uniform(0, 1)
        y = random.uniform(0, 1)
    
        if x*x + y*y < 1.:
            h += 1
    
    return 4. * h / n

@jit(nopython=True, nogil=True, parallel=True)
def calc_pi_parallel(n):
    h = 0
    for _ in prange(n):
        x = random.uniform(0, 1)
        y = random.uniform(0, 1)
        if x**2 + y**2 < 1:
            h += 1
    return 4. * h / n


In [12]:
import os
os.cpu_count()

128

In [49]:
# Selection of the function to run and the number of iterations

def mem_time_calc(n, func, logs=False):

    if func == 'python':
        f = lambda: calc_pi_loop(int(n))

    elif func == 'numpy':
        f = lambda: calc_pi_numpy(int(n))
        
    elif func == 'numba':
        f = lambda: calc_pi_numba(int(n))
        
    elif func == 'numba_parallel':
        f = lambda: calc_pi_parallel(int(n))
    
    
    start = time.perf_counter() 
    max_mem, pi_est = mmu(f, max_usage=True, retval=True)
    elapsed = time.perf_counter() - start
    
    if logs is not None and logs:
        print(f"N = {n:0.0e}, pi = {pi_est:0.4e}, time = {elapsed:0.4e}s, max_memory={max_mem}MiB")
        print()
        
    return pi_est, elapsed, max_mem    

    

In [13]:


type = ['python', 'numpy', 'numba', 'numba_parallel']

N = [1e3, 1e4, 1e5, 1e6, 1e7, 1e8, 1e9]

df = pd.DataFrame(columns=['method', 'N', 'pi_mean', 'pi_std', 'pi_abs_error', 'runtime', 'time_per_sec', 'memory_used', 'sample_per_sec'])

for t in type:
    

    for n in tqdm(N):
        
        pi_n = []
        elapsed_n = []
        max_mem_n = []
        
        for i in tqdm(range(10)):
            
            if (t != 'numba_parallel'):
                #if (n > 1e8):
                #    break
                
                if (i > 0):
                    break
            
            a, b, c = mem_time_calc(n, func = t, logs=False)
            
            pi_n.append(a)
            elapsed_n.append(b)
            max_mem_n.append(c)
        
        if len(pi_n) == 0:
            continue
        
        runtime = sum(elapsed_n)/len(elapsed_n)
        time_per_iter = runtime/n
        sample_per_sec = n/runtime   
        memory_used = sum(max_mem_n)/len(max_mem_n)    
    
        pi_mean = sum(pi_n)/len(pi_n)
        pi_abs_error = sum([abs(x - pi_mean) for x in pi_n])/len(pi_n)
        pi_std = np.std(pi_n)
        
        results = {'method':t,
                   'N':n,
                   'pi_mean':pi_mean,
                   'pi_std':pi_std,
                   'pi_abs_error':pi_abs_error,
                   'runtime':runtime,
                   'time_per_sec':time_per_iter,
                   'memory_used':memory_used,
                   'sample_per_sec':sample_per_sec
                   }
        
        df = pd.concat([df, pd.DataFrame([results])], ignore_index=True)
        
    print(f'Completed calculation using method: {t}')
            
df.to_csv('all_pi_estimation_results.csv', index=False)

 10%|█         | 1/10 [00:00<00:01,  8.19it/s]
/glade/derecho/scratch/prasoonv/tmp/ipykernel_130113/4045638171.py:54: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([results])], ignore_index=True)
100%|██████████| 7/7 [08:20<00:00, 71.48s/it] 


Completed calculation using method: python


 86%|████████▌ | 6/7 [00:09<00:02,  2.73s/it]

: 

In [41]:
path = '/glade/work/prasoonv/hpc_course/hw3/'

all_data = pd.read_csv(path + 'all_pi_estimation_results.csv')
data_cpp = pd.read_csv(path + 'results_cpp.csv')
data_cython = pd.read_csv(path + 'cython/results_cython.csv')

all_data = pd.concat([all_data, data_cpp, data_cython], ignore_index=True)

print(all_data.tail(10))

    method             N   pi_mean    pi_std  pi_abs_error    runtime  \
32     c++  1.000000e+07  3.141390  0.000384      0.002871   0.225940   
33     c++  1.000000e+08  3.141590  0.000252      0.002162   2.259090   
34     c++  1.000000e+09  3.141570  0.000049      0.000358  22.620300   
35  cython  1.000000e+03  3.159200  0.046331      0.036800   0.079518   
36  cython  1.000000e+04  3.149680  0.017056      0.014880   0.078857   
37  cython  1.000000e+05  3.140040  0.003707      0.002856   0.085040   
38  cython  1.000000e+06  3.141416  0.002099      0.001793   0.072287   
39  cython  1.000000e+07  3.141335  0.000570      0.000500   0.237820   
40  cython  1.000000e+08  3.141611  0.000123      0.000098   1.067524   
41  cython  1.000000e+09  3.141599  0.000047      0.000042  10.613506   

    time_per_sec  memory_used  sample_per_sec  
32  2.259400e-08       2.0000    4.425960e+07  
33  2.259090e-08       2.0000    4.426560e+07  
34  2.262030e-08       2.0000    4.420800e+07  
35  

In [24]:
avg_runtime_numba_all = all_data[all_data['method'] == 'numba'].groupby('N')['runtime'].mean()
avg_runtime_numba_parallel_all = all_data[all_data['method'] == 'numba_parallel'].groupby('N')['runtime'].mean()


print('Speedup of Numba Parallel over Numba:', avg_runtime_numba_all/avg_runtime_numba_parallel_all)

Speedup of Numba Parallel over Numba: N
1.000000e+03    36.651661
1.000000e+04     0.749219
1.000000e+05     0.894536
1.000000e+06     0.831435
1.000000e+07     2.196351
1.000000e+08     2.371367
1.000000e+09     2.394153
Name: runtime, dtype: float64


In [43]:

name_all = ['Python Loop', 'Numpy', 'Numba', 'Numba Parallel', 'Cython', 'C++']
types = ['python', 'numpy', 'numba', 'numba_parallel', 'cython', 'c++']

key = ['memory_used', 'pi_mean', 'runtime', 'sample_per_sec']
name = ['Memory (MiB) Usage', 'Pi Value', 'Time Elapsed (s)', 'Samples per Second']

for k, n in zip(key, name):
    
    #d = all_data.pivot(index='N', columns='method', values=k).reset_index()

    # Create figure 
    fig = go.Figure()

    # Add traces
    for i, t in enumerate(types):
        d_i = all_data[all_data['method'] == t]
        fig.add_trace(go.Scatter(x=d_i['N'], y=d_i[k], mode='lines+markers', name=name_all[i]))
    
    if k == 'pi_mean':
        fig.add_trace(go.Scatter(x=d_i['N'], y=[np.pi]*6, mode='lines', name='True Pi Value', line=dict(dash='dash')))
    
    #fig.update_xaxes(range=[10**3, 10**9])    

    # Add layout
    fig.update_xaxes(type='log', range=[3,9])
    if k == 'runtime':
        fig.update_yaxes(type='log')
        
    if k == 'pi_mean':
        fig.update_yaxes(range=[3.1,3.16])

    fig.update_layout(
        title=f'{n} Plots',
        xaxis_title='N',
        yaxis_title=n,
        template='plotly_dark',
    )

    fig.show()

# Working with Fastest Method Numba Parallel

In [51]:
t = 'numba_parallel'

N = [1e3, 1e4, 1e5, 1e6, 1e7, 1e8, 1e9, 1e10, 1e11]

df = pd.DataFrame(columns=['method', 'N', 'pi_mean', 'pi_std', 'pi_abs_error', 'runtime', 'time_per_sec', 'memory_used', 'sample_per_sec'])

for n in tqdm(N):
    
    pi_n = []
    elapsed_n = []
    max_mem_n = []
    
    for i in tqdm(range(10)):
                
        a, b, c = mem_time_calc(n, func = t, logs=False)
        
        pi_n.append(a)
        elapsed_n.append(b)
        max_mem_n.append(c)
    
    if len(pi_n) == 0:
        continue
    
    runtime = sum(elapsed_n)/len(elapsed_n)
    time_per_iter = runtime/n
    sample_per_sec = n/runtime   
    memory_used = sum(max_mem_n)/len(max_mem_n)    

    pi_mean = sum(pi_n)/len(pi_n)
    pi_abs_error = sum([abs(x - pi_mean) for x in pi_n])/len(pi_n)
    pi_std = np.std(pi_n)
    
    results = {'method':t,
                'N':n,
                'pi_mean':pi_mean,
                'pi_std':pi_std,
                'pi_abs_error':pi_abs_error,
                'runtime':runtime,
                'time_per_sec':time_per_iter,
                'memory_used':memory_used,
                'sample_per_sec':sample_per_sec
                }
    
    df = pd.concat([df, pd.DataFrame([results])], ignore_index=True)
    
print(f'Completed calculation using method: {t}')
        
df.to_csv('numba_parallel_results.csv', index=False)

100%|██████████| 10/10 [00:00<00:00, 10.80it/s]
/glade/derecho/scratch/prasoonv/tmp/ipykernel_12297/356474777.py:44: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.

100%|██████████| 9/9 [21:47<00:00, 145.32s/it]

Completed calculation using method: numba_parallel


In [1]:
from numpy.random import rand
import sys
import numpy as np
from tqdm import tqdm
from memory_profiler import memory_usage as mmu
import time

import random, sys
from numba import jit, njit, prange

import xarray as xr
import plotly.express as py
import pandas as pd

import plotly.graph_objects as go

In [12]:
numba_data = pd.read_csv('numba_parallel_results.csv')

print(numba_data.head(10))

           method             N   pi_mean    pi_std  pi_abs_error     runtime  \
0  numba_parallel  1.000000e+03  3.156800  0.040032      0.034560    0.090992   
1  numba_parallel  1.000000e+04  3.138000  0.011851      0.010400    0.087823   
2  numba_parallel  1.000000e+05  3.140620  0.003345      0.003008    0.086466   
3  numba_parallel  1.000000e+06  3.141474  0.001504      0.001224    0.092280   
4  numba_parallel  1.000000e+07  3.141129  0.000646      0.000538    0.122120   
5  numba_parallel  1.000000e+08  3.141637  0.000145      0.000121    0.288264   
6  numba_parallel  1.000000e+09  3.141589  0.000069      0.000058    1.129407   
7  numba_parallel  1.000000e+10  3.141594  0.000020      0.000016   11.022196   
8  numba_parallel  1.000000e+11  3.141590  0.000005      0.000003  117.847541   

   time_per_sec  memory_used  sample_per_sec  
0  9.099183e-05   289.066406    1.099000e+04  
1  8.782284e-06   289.066406    1.138656e+05  
2  8.646553e-07   289.066406    1.156530e+06  
3

In [13]:
# Use the numba_parallel dataset collected in this notebook
plot_data = numba_data.sort_values('N')

n_vals = plot_data['N'].astype(float).to_numpy()
mean_abs_error = plot_data['pi_abs_error'].to_numpy()
std_vals = plot_data['pi_std'].to_numpy()



In [14]:
fig_err_std = go.Figure()
fig_err_std.add_trace(go.Scatter(
    x=n_vals, y=mean_abs_error, mode='lines+markers', name='|pi_est - π|'
))
fig_err_std.add_trace(go.Scatter(
    x=n_vals, y=std_vals, mode='lines+markers', name='standard deviation'
))

fig_err_std.update_xaxes(type='log', title='n', range=[3,11])
fig_err_std.update_yaxes(type='log', title='Error / Standard Deviation')
fig_err_std.update_layout(
    title='Mean Absolute Error and Standard Deviation vs n',
    template='plotly_dark'
)
fig_err_std.show()


In [18]:
x = n_vals
y = std_vals

logx = np.log10(x)
logy = np.log10(y)

alpha, intercept = np.polyfit(logx, logy, 1)
alpha *= -1

A = np.exp(intercept)
print(f"Estimated power law: std ~ {A:.2e} * n^{-alpha:.2f}")

n_fit = x
std_fit = A * n_fit**(-alpha)


Estimated power law: std ~ 1.00e+00 * n^-0.47


In [24]:
fig_fit = go.Figure()
fig_fit.add_trace(go.Scatter(
    x=x, y=y, mode='markers+lines', name='Observed σ'
))
fig_fit.add_trace(go.Scatter(
    x=n_fit, y=std_fit, mode='lines',
    name=f'Fit: σ = A n^(-α), α={alpha:.4f}'
))

fig_fit.update_xaxes(type='log', title='n')
fig_fit.update_yaxes(type='log', title='σ')
fig_fit.update_layout(
    title='Power-law Fit of Standard Deviation vs n',
    template='plotly_dark'
)
fig_fit.show()


In [33]:
alpha_the = 0.5
relative_err = abs(alpha - alpha_the) / abs(alpha_the)

print(f'{alpha:0.3e} +- {relative_err*100:.2f}% relative error compared to theoretical value of 0.5')

4.741e-01 +- 5.18% relative error compared to theoretical value of 0.5


# Equation for n to get $10^{-12}$ precision in $\pi$: 
since, $$\sigma=An^{-\alpha}$$

For $10^{-12}$ precision of $\pi$, the value of $\sigma$ should be at max $10^{-13}$.

Plugging this value in above, we can get the value of `n`.

In [44]:
sigma = 10**-13
n = (1/sigma) ** (1/alpha)

print(f"To achieve a standard deviation of {sigma:.1e}, we need approximately n = {n:0.2e} samples.")

To achieve a standard deviation of 1.0e-13, we need approximately n = 2.63e+27 samples.


In [ ]:
sample_per_sec_numba_parallel = 1.5*10**8

print(f"With a sampling rate of {sample_per_sec_numba_parallel:0.2e} samples/sec of Numba parallel,\nit would take approximately {n/(sample_per_sec_numba_parallel*86400*365.25):0.2e} years to achieve this precision.")



With a sampling rate of 1.50e+08 samples/sec of Numba parallel,
it would take approximately 5.57e+11 years to achieve this precision.


In [56]:
#print(numba_data.head(10))

sample_per_sec_fastest = numba_data['sample_per_sec'].max()
time_per_core = n / sample_per_sec_fastest

cores_1_year_compute = time_per_core / (86400*365.25)

print(f"With the fastest sampling rate of {sample_per_sec_fastest:0.2e} samples/sec of Numba parallel,\nit would take approximately {time_per_core/(86400*365.25):0.2e} years on a single core to achieve this precision.\nTo achieve this in 1 year, we would need approximately {cores_1_year_compute:0.2e} cores.")

With the fastest sampling rate of 9.07e+08 samples/sec of Numba parallel,
it would take approximately 9.20e+10 years on a single core to achieve this precision.
To achieve this in 1 year, we would need approximately 9.20e+10 cores.
